# Modelos no lineales (2): TVP-VAR con volatilidad estocástica

El Threshold VAR del notebook A cambia de golpe entre dos regímenes. La segunda familia de modelos
no lineales cambia **despacio y todo el tiempo**: cada coeficiente es un paseo aleatorio y cada
varianza también. Es el TVP-VAR-SV de Primiceri (2005), escrito en la forma estructural de Chan y
Eisenstat (2018), que es la que estima `MacroPy`:

$$
A_t\,\mathbf{y}_t = \boldsymbol{\mu}_t + B_{1,t}\mathbf{y}_{t-1} + B_{2,t}\mathbf{y}_{t-2} + \boldsymbol{\varepsilon}_t,
\qquad \boldsymbol{\varepsilon}_t \sim N(0, \Sigma_t),\quad \Sigma_t = \mathrm{diag}(e^{h_{1t}}, \ldots, e^{h_{nt}})
$$

$$
\theta_t = \theta_{t-1} + \eta_t, \qquad h_t = h_{t-1} + \zeta_t,
$$

con $A_t$ triangular inferior con unos en la diagonal (la identificación recursiva de la sesión 3,
fecha por fecha) y $\theta_t$ el vector con los interceptos, los coeficientes de los rezagos y los
elementos libres de $A_t$. El punto de partida es la comparación más simple: el VAR de coeficientes
y varianzas constantes, `C`, contra el modelo completo, `TVP-SV`. Después entran dos competidores
intermedios de la familia de Chan y Eisenstat.

1. **Datos, exogeneidad de bloque y dos modelos**: `C` y `TVP-SV`, con su verosimilitud marginal y
   su DIC.
2. **Volatilidad estocástica frente a varianza constante.**
3. **Impulso-respuesta**: la mediana de las medianas, superficies interactivas y fechas específicas.
4. **Descomposición de varianza a 20 trimestres e histórica.**
5. **Competencia**: `C-SV` y `TVP-R1-SV`. ¿Qué modelo gana?


## 1. Datos, exogeneidad de bloque y dos modelos

El sistema es el de la economía pequeña y abierta de las sesiones anteriores: crecimiento de los
términos de intercambio, del PBI e inflación, todos interanuales, y la tasa interbancaria,
trimestral, de 1997Q1 a 2026Q2.

In [ ]:
import os
for var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):
    os.environ.setdefault(var, "1")         # antes de numpy: ver la nota técnica de abajo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import TimeVaryingVAR, compare_tvpvar, generate_series_plot, plot_irf_over_time, set_style
from MacroPy.style import PALETTE
import sys; sys.path.append("..")          # course_data.py vive en la raíz del repositorio
import course_data as cd

set_style()
levels = cd.load()                          # copia congelada de la base del curso
Y = cd.small_open_economy(levels, start="1997-03-01", end="2026-06-01")
names = list(Y.columns)
titles = [cd.LABELS[c] for c in names]
shocks = ["externo", "actividad", "precios", "política"]
print(f"Muestra: {cd.quarter(Y.index[0])} a {cd.quarter(Y.index[-1])} ({len(Y)} trimestres)")
generate_series_plot(Y, series_titles=titles, title="Perú, 1997-2026: la pandemia adentro")

El PBI cae 30 % interanual en 2020Q2 y sube 42 % un año después. En la sesión 2 esos trimestres
pedían dummies o el escalamiento de Lenza y Primiceri. Aquí la varianza $e^{h_t}$ puede subir en
2020 y volver a bajar: la pandemia queda adentro, como un pico de varianza.

**Exogeneidad de bloque.** El Perú no mueve los precios internacionales. Los términos de intercambio
van primero en el orden recursivo, así que ningún choque doméstico los mueve al impacto; con
`b_exo[0, 1:] = 1` tampoco dependen de los rezagos domésticos, de modo que ningún choque doméstico
los mueve en ningún horizonte, en ninguna fecha. En `TimeVaryingVAR` los coeficientes excluidos se
fijan en cero en los bloques que derivan y en los constantes; la restricción se puede probar con
un factor de Bayes (sección 5).

`TimeVaryingVAR` estima cualquier miembro de la familia con el argumento `model`: `"C"` (todo
constante) y `"TVP-SV"` (todo cambia). `presample=4` reserva los primeros trimestres como
condiciones iniciales comunes a todos los modelos, requisito para comparar sus verosimilitudes
marginales. Los priors son los del paper (`TVP_PRIOR_DEFAULTS`): poco informativos sobre los
niveles iniciales y conservadores sobre cuánto puede cambiar cada coeficiente por trimestre.

**Nota técnica: hilos y procesos.** Las primeras líneas de la celda anterior dejan el álgebra
lineal de `numpy` con un solo hilo (OpenBLAS en Windows, Linux, Codespaces y Mac con Intel; MKL en
Anaconda). Las matrices de estos modelos son pequeñas y coordinar muchos hilos cuesta más que la
cuenta misma: en un contenedor Linux de diez núcleos, una evaluación de la verosimilitud del
TVP-SV toma 0.1 segundos con un hilo y 20 con todos. El paralelismo útil es otro: `n_jobs=N_JOBS`
reparte evaluaciones completas entre hasta cuatro procesos, de unos 200 MB de memoria cada uno.
Las líneas tienen efecto solo antes de importar `numpy`: si el kernel ya lo importó, hay que
reiniciarlo. En un Mac con chip Apple no cambian nada, porque `numpy` usa Accelerate.

In [ ]:
b_exo = np.zeros((4, 4), dtype=int)
b_exo[0, 1:] = 1              # fila = ecuación: los términos de intercambio excluyen los rezagos domésticos

N_JOBS = min(4, os.cpu_count() or 1)       # procesos para la verosimilitud marginal y el DIC

def estimate(spec, data=Y, block=True, seed=1):
    m = TimeVaryingVAR(data, model=spec, lags=2, presample=4, post_draws=15000, burnin=0.2,
                       hor=21, seed=seed, progress=False, b_exo=b_exo if block else None)
    m.sample_posterior()
    return m

models = {spec: estimate(spec) for spec in ["C", "TVP-SV"]}
tvp, cvar = models["TVP-SV"], models["C"]
print(f"{tvp.T} observaciones efectivas; {tvp.draws['save_idx'].shape[0]} trayectorias de estados guardadas")
tvp.model_summary()

### Verosimilitud marginal y DIC

Dos criterios, los de Chan y Eisenstat (2018), ambos sobre la **verosimilitud integrada**, con los
estados $\theta_{1:T}$ y $h_{1:T}$ integrados:

* La **verosimilitud marginal** $p(\mathbf{y})$ promedia la verosimilitud sobre el prior: premia el
  ajuste y castiga la flexibilidad que no se usa. Se estima por muestreo de importancia, con una
  densidad ajustada a las extracciones posteriores de los hiperparámetros. Mayor es mejor.
* El **DIC** es el desvío esperado más dos veces el número efectivo de parámetros, $p_D$. Menor es
  mejor.

Con 500 extracciones de importancia para la verosimilitud marginal y una de cada 40 extracciones
retenidas para el DIC, cada una de las dos celdas siguientes toma menos de un minuto en un
portátil reciente con dos núcleos; casi todo el tiempo es del `TVP-SV`. La columna `nse` es el error numérico; entre modelos con varios
cientos de puntos de diferencia, no importa.

In [ ]:
for m in models.values():
    m.log_marginal_likelihood(n_draws=500, n_jobs=N_JOBS)    # 500 extracciones de importancia

In [ ]:
for m in models.values():
    m.dic(simstep=40, n_jobs=N_JOBS)                         # una de cada 40 extracciones retenidas: 300
compare_tvpvar(models).round(1)

El modelo completo gana por **más de 200 puntos** de log verosimilitud marginal y por 370 de
DIC. No es una diferencia que dependa de detalles: el VAR de varianza constante no puede con la
pandemia ni con el cambio de régimen monetario de comienzos de los 2000. La pregunta interesante
viene en la sección 5: ¿qué parte de la ganancia viene de la volatilidad y qué parte de los
coeficientes cambiantes?

## 2. Volatilidad estocástica frente a varianza constante

La desviación estándar de cada choque estructural, trimestre a trimestre, con su banda al 68 %, y
la del modelo `C`, constante por construcción.

In [ ]:
vol_tvp, vol_c = tvp.volatility_frame(cred=0.68), cvar.volatility_frame(cred=0.68)
fig, axes = plt.subplots(2, 2, figsize=(11, 6.4), sharex=True, constrained_layout=True)
for ax, var, title in zip(axes.ravel(), names, titles):
    v, c = vol_tvp[var], vol_c[var]
    ax.fill_between(v.index, v["lower"], v["upper"], color=PALETTE["accent"], alpha=0.18, lw=0,
                    label="TVP-SV, banda al 68 %")
    ax.plot(v.index, v["median"], color=PALETTE["accent"], lw=1.8, label="TVP-SV, mediana")
    ax.fill_between(c.index, c["lower"], c["upper"], color=PALETTE["navy"], alpha=0.15, lw=0)
    ax.plot(c.index, c["median"], "--", color=PALETTE["navy"], lw=1.6, label="C, constante")
    ax.set_title(title)
    ax.set_ylim(bottom=0)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Desviación estándar de los choques estructurales: volatilidad estocástica frente a varianza constante")
plt.show()
ratio = (vol_tvp.xs("median", axis=1, level=1).max() / vol_tvp.xs("median", axis=1, level=1).min()).round(1)
print("Razón entre la desviación estándar máxima y la mínima en el TVP-SV:", ratio.to_dict())
print("Desviación estándar constante del modelo C:", vol_c.xs("median", axis=1, level=1).iloc[0].round(2).to_dict())

La varianza constante es un promedio que no describe ningún período. La desviación estándar del
choque al PBI se multiplica por más de diez en 2020-21 y vuelve en 2022 al nivel de la década
anterior; el modelo `C` la fija en 5.0 en toda la muestra, cuatro veces la de un trimestre típico
fuera de la pandemia (1.2). La del choque de tasa cae de siete puntos en 1998 a menos de una décima desde 2003, con las
metas de inflación; `C` le asigna 1.8. La de los términos de intercambio sube hasta 2009-10, baja
con el fin del superciclo y vuelve a subir al final de la muestra. Son estimaciones **suavizadas**:
usan toda la muestra, incluida información posterior a cada fecha, por eso la del PBI empieza a
subir a fines de 2019.

## 3. Impulso-respuesta

Con $A_t^{-1}\Sigma_t^{1/2}$ como factor de Cholesky de cada fecha, `compute_irfs` calcula una
respuesta por trimestre: la mediana posterior y las bandas para todas las fechas, y las
extracciones completas solo para las fechas pedidas. Hay entonces 114 respuestas, una por
trimestre. Un resumen útil es la **mediana de las medianas**: en cada horizonte, la mediana entre
fechas de la mediana posterior de cada fecha, con la mediana entre fechas de sus percentiles 16 y
84 como banda. Se compara con la respuesta del modelo `C`, que es la misma en todas las fechas.

In [ ]:
fechas = ["2008Q4", "2015Q4", "2024Q4"]
for m in models.values():
    m.compute_irfs(dates=fechas, cred_interval=0.68, hor=21)

def summary_irf(m):
    """Mediana entre fechas de la mediana posterior y de los percentiles 16 y 84 de cada fecha."""
    lo, hi = m.irf["bands"][0.68]
    return np.median(m.irf["median"], axis=0), np.median(lo, axis=0), np.median(hi, axis=0)

def plot_summary(shock, title):
    s = names.index(shock)
    tv, cc = summary_irf(tvp), summary_irf(cvar)
    steps = np.arange(tv[0].shape[0])
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.2), constrained_layout=True)
    for ax, k, t in zip(axes.ravel(), range(len(names)), titles):
        ax.fill_between(steps, tv[1][:, k, s], tv[2][:, k, s], color=PALETTE["accent"], alpha=0.18, lw=0,
                        label="TVP-SV, mediana de medianas y banda al 68 %")
        ax.plot(steps, tv[0][:, k, s], color=PALETTE["accent"], lw=1.8)
        ax.fill_between(steps, cc[1][:, k, s], cc[2][:, k, s], color=PALETTE["navy"], alpha=0.12, lw=0)
        ax.plot(steps, cc[0][:, k, s], "--", color=PALETTE["navy"], lw=1.6, label="C, mediana y banda al 68 %")
        ax.axhline(0, color="gray", lw=0.7)
        ax.set_title(t)
        ax.set_xlabel("trimestres")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

plot_summary("tot", "Respuesta a un choque de términos de intercambio de una desviación estándar")

La forma es la misma en los dos modelos: un choque externo favorable sube el PBI, con el máximo
entre cuatro y cinco trimestres después, y baja la inflación durante el primer año. Las magnitudes no: el modelo
`C` da una respuesta del PBI dos veces mayor y una de la tasa más persistente. Con varianza
constante, el tamaño del choque y su transmisión salen de un promedio que incluye la pandemia; en
el `TVP-SV` cada fecha tiene su propio choque y su propia economía, y las bandas son más anchas
porque también lo es la incertidumbre sobre los coeficientes.

In [ ]:
plot_summary("i", "Respuesta a un choque de política monetaria de una desviación estándar")

Con el choque de política la diferencia es de tamaño, no de transmisión. En `C`, una desviación
estándar del choque de tasa es 1.8 puntos: el tamaño típico de 1997-2002, cuando la tasa
interbancaria saltaba. En el `TVP-SV`, la mediana entre fechas es de una décima, porque desde 2003
casi todo lo que hace el banco central es sistemático (sesión 3) y lo que queda como sorpresa es
pequeño. **Una desviación estándar no es un tamaño comparable entre modelos ni entre fechas.**

### Superficies interactivas

La mediana de las medianas resume las fechas; las superficies las muestran todas. La función
siguiente dibuja, con `plotly`, la respuesta mediana de cada variable al choque externo por fecha
y horizonte: cada superficie se puede **rotar, acercar y leer punto por punto** con el cursor (en
Jupyter, VS Code y Codespaces; no en la vista estática de GitHub). Los colores son los de
`MacroPy`: azul para respuestas negativas y terracota para positivas, centrados en cero.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.colors import to_hex
from MacroPy.style import colormap


def superficies_3d(model, shock, titles, scale=1.0, title=None):
    """Respuesta mediana de cada variable a `shock`, por fecha (eje y) y horizonte (eje x)."""
    j = model.names.index(shock)
    med = model.irf["median"][:, :, :, j] * scale               # (fecha, horizonte, variable)
    fechas = pd.DatetimeIndex(model.irf["dates"])
    anios = fechas.year + (fechas.month - 1) / 12
    horizonte = np.arange(med.shape[1])
    cmap = colormap("diverging")
    escala = [[x, to_hex(cmap(x))] for x in np.linspace(0, 1, 11)]
    n = len(titles)
    cols = 2 if n == 4 else min(n, 3)
    rows = -(-n // cols)
    fig = make_subplots(rows=rows, cols=cols, specs=[[{"type": "scene"}] * cols] * rows,
                        subplot_titles=titles, horizontal_spacing=0.02, vertical_spacing=0.06)
    for k in range(n):
        z = med[:, :, k]
        tope = float(np.abs(z).max()) or 1.0
        fig.add_trace(go.Surface(x=horizonte, y=anios, z=z, colorscale=escala, cmin=-tope, cmax=tope,
                                 showscale=False, opacity=0.95,
                                 hovertemplate="año %{y:.2f}<br>horizonte %{x}<br>respuesta %{z:.2f}<extra></extra>"),
                      row=k // cols + 1, col=k % cols + 1)
    fig.update_scenes(xaxis_title="horizonte", yaxis_title="fecha", zaxis_title="pp",
                      camera=dict(eye=dict(x=1.7, y=-1.5, z=0.9)), aspectmode="manual",
                      aspectratio=dict(x=1, y=1.6, z=0.7))
    fig.update_layout(height=430 * rows, width=1000, title=title, margin=dict(l=0, r=0, t=60, b=0))
    return fig


superficies_3d(tvp, "tot", titles,
               title="TVP-SV: respuestas a un choque de términos de intercambio de una desviación estándar").show()

### Respuestas en fechas específicas

Tres fechas: la crisis financiera global (2008Q4), el fin del superciclo (2015Q4) y el final de la
muestra (2024Q4). Y la respuesta del PBI a dos, cuatro y ocho trimestres a lo largo de toda la
muestra.

In [ ]:
tvp.plot_irfs(shock="tot", cred=0.68, series_titles=titles,
              title="TVP-SV: respuesta a un choque de términos de intercambio en tres fechas (banda al 68 %)")
plot_irf_over_time(tvp, response="y", shock="tot", horizons=(2, 4, 8), cred=0.68)

In [ ]:
def respuesta_acumulada(model, shock, response, fechas, horizonte=8, scale=1.0):
    """Suma de la respuesta mediana entre el impacto y `horizonte` trimestres, por fecha."""
    t = [[cd.quarter(d) for d in model.yy_dates].index(f) for f in fechas]
    med = model.irf["median"][t, :horizonte + 1, names.index(response), names.index(shock)]
    return med.sum(axis=1) * scale

vol_tot = {cd.quarter(d): v for d, v in vol_tvp["tot"]["median"].items()}
tabla = {"desv. estándar del choque externo": [vol_tot[f] for f in fechas],
         "PBI acumulado a 8 trim., choque de 1 d.e.": respuesta_acumulada(tvp, "tot", "y", fechas)}
tvp.irf_1std = 0                                    # choque unitario: mismo tamaño en cada fecha
tvp.compute_irfs(dates=fechas, cred_interval=0.68, hor=21)
tabla["PBI acumulado a 8 trim., choque de +10 pp"] = respuesta_acumulada(tvp, "tot", "y", fechas, scale=10)
tvp.irf_1std = 1                                    # de vuelta al choque de una desviación estándar
tvp.compute_irfs(dates=fechas, cred_interval=0.68, hor=21)
pd.DataFrame(tabla, index=fechas).round(2)

El choque externo pesó más alrededor de la crisis financiera global: la respuesta acumulada del
PBI a dos años a un choque de una desviación estándar es 4.6 puntos en 2008Q4, 0.8 en 2015Q4 y 1.8
en 2024Q4. Pero esa desviación estándar también cambia, 6.6 puntos en 2008 frente a 4.5 en 2015 y
6.2 en 2024: una respuesta mayor puede venir de un choque más grande o de una economía más
sensible. Con el mismo choque de diez puntos en todas las fechas (`irf_1std = 0`), la respuesta es
7.1 en 2008Q4, 1.8 en 2015Q4 y 2.9 en 2024Q4. La transmisión sí fue mayor en 2008: 2.4 veces la
de 2024 y 3.9 veces la de 2015, menos que las 5.8 veces que sugería el choque de una desviación
estándar frente a 2015. **Antes de comparar
fechas, hay que fijar el tamaño del choque.**

## 4. Descomposición de varianza e histórica

Con coeficientes cambiantes, la FEVD también cambia: `compute_fevd` devuelve la participación de
cada choque por fecha y horizonte. Se reporta a **20 trimestres**, el largo plazo: cuando el
horizonte crece, la FEVD de un VAR estacionario converge a la descomposición de la varianza
incondicional. En un TVP-VAR esa varianza es la del VAR con los parámetros de cada fecha, congelados
hacia adelante. La celda siguiente comprueba que entre 20 y 40 trimestres las participaciones casi
no cambian. La FEVD siempre usa choques de una desviación estándar, sea cual sea `irf_1std`.

In [ ]:
fevd = tvp.compute_fevd(hor=41)["fevd"]               # (fecha, horizonte, choque, variable)
H = 20                                                  # entrada H - 1: la varianza del error a H pasos
cambio = np.abs(fevd[:, 2 * H - 1] - fevd[:, H - 1]).max()
print(f"Mayor cambio de una participación entre 20 y 40 trimestres, en cualquier fecha: {cambio:.1f} pp")
tvp.plot_fevd(horizon=H, series_titles=titles, shock_titles=shocks,
              title="TVP-SV: descomposición de la varianza del error de pronóstico a 20 trimestres")

In [ ]:
fevd_c = cvar.compute_fevd(hor=H)["fevd"]
rows = {}
for spec, fe in (("C", fevd_c), ("TVP-SV", fevd[:, :H])):
    for date in fechas:
        t = [cd.quarter(d) for d in tvp.yy_dates].index(date)
        rows[(spec, date)] = fe[t, H - 1, :, names.index("y")]
pd.DataFrame(rows, index=shocks).T.round(1).rename_axis(["modelo", "fecha"]) \
    .rename(columns=lambda s: f"choque {s}")

Los modelos discrepan sobre **cuánto y cuándo** importa el exterior. A 20 trimestres, el choque
externo explica 10 % de la varianza del PBI en el modelo `C`, el mismo número en todas las fechas.
En el `TVP-SV` explica 47 % en 2008Q4, 51 % en 2015Q4 y 66 % en 2024Q4: con la volatilidad del
choque al PBI de vuelta en su nivel bajo y la de los términos de intercambio en alza, el exterior
gana peso. El modelo `C` reparte la varianza de la pandemia en toda la muestra y se la atribuye al
choque de actividad.

La descomposición histórica reparte cada trimestre observado entre la base y los choques
acumulados, con la identidad contable exacta en cada extracción.

In [ ]:
hd = tvp.compute_hd(plot_hd=True, series_titles=titles, shock_titles=shocks)
print("máx |base + aportes - dato| =", f"{np.abs(hd['baseline'] + hd['shock'].sum(axis=1) - tvp.yy).max():.1e}")

In [ ]:
episodios = {"crisis financiera 2008Q3-2009Q4": ("2008-09-01", "2009-12-01"),
             "pandemia 2020": ("2020-03-01", "2020-12-01"),
             "inflación 2022-2023": ("2022-03-01", "2023-12-01")}
rows = {}
for spec, m in models.items():
    h = m.compute_hd()
    d = pd.DatetimeIndex(h["dates"])
    for name, (a, b) in episodios.items():
        mask = (d >= a) & (d <= b)
        rows[(spec, name)] = {"PBI observado": m.yy[mask, 1].mean(), "base": h["baseline"][mask, 1].mean(),
                              **{f"aporte {s}": h["shock"][mask, k, 1].mean() for k, s in enumerate(shocks)}}
pd.DataFrame(rows).T.round(2).rename_axis(["modelo", "episodio"])

En la pandemia los dos modelos coinciden: la caída es casi toda un choque de actividad. En
2008-09 también: el exterior resta dos puntos y medio al crecimiento en ambos. En 2022-23 difieren:
el modelo `C` reparte el menor crecimiento entre choques externos, de actividad y de precios, y le
atribuye un aporte positivo a la política; el `TVP-SV` pone casi todo en una **base más baja**, 2.9
frente a 4.5, es decir, en un crecimiento tendencial que viene cayendo y que un intercepto
constante no puede capturar. Las descomposiciones son sensibles a la especificación justo en los
episodios que uno quiere explicar.

## 5. Competencia: C-SV y TVP-R1-SV

¿La ventaja del `TVP-SV` viene de la volatilidad o de los coeficientes cambiantes? Chan y
Eisenstat (2018) proponen una familia que apaga una fuente de variación a la vez. Dos miembros
compiten aquí:

* `C-SV`: coeficientes constantes y volatilidad estocástica.
* `TVP-R1-SV`: solo derivan los elementos libres de $A_t$ (las relaciones contemporáneas) y las
  volatilidades; los interceptos y los rezagos son constantes. La transmisión al impacto puede
  cambiar en el tiempo, la propagación posterior no.

Otros dos miembros quedan para los ejercicios: `TVP-R2-SV`, en el que derivan los interceptos y los
rezagos, y `TVP-R3-SV`, en el que solo derivan los interceptos. Una advertencia sobre la R3: como
$A$ y los rezagos son constantes, sus respuestas a **choques unitarios** son las mismas en todas
las fechas; las de una desviación estándar cambian solo porque cambia el tamaño del choque. Una
respuesta de la R3 que cambia en el tiempo con choques de una desviación estándar no es un cambio
en la transmisión.

In [ ]:
for spec in ["C-SV", "TVP-R1-SV"]:
    models[spec] = estimate(spec)
    models[spec].log_marginal_likelihood(n_draws=500, n_jobs=N_JOBS)
    models[spec].dic(simstep=40, n_jobs=N_JOBS)
models = {spec: models[spec] for spec in ["C", "C-SV", "TVP-R1-SV", "TVP-SV"]}
compare_tvpvar(models).round(1)

Gana el **`C-SV`** en los dos criterios: $-997.8$ de log verosimilitud marginal, 3.9 puntos por
encima del `TVP-R1-SV` (un factor de Bayes cercano a 50) y casi 12 por encima del `TVP-SV`; el DIC da el
mismo orden. Casi toda la ganancia frente al modelo `C`, más de 200 puntos, viene de la volatilidad
estocástica; los coeficientes cambiantes no pagan su costo en estos datos, y entre ellos la R1, que
solo deja cambiar $A_t$, es la que menos pierde. Es la conclusión de Chan y Eisenstat (2018) con
datos de Estados Unidos: el `CVAR-SV` supera al TVP-SV completo. Las diferencias entre `C-SV` y
`TVP-R1-SV` son de pocos puntos, y en los modelos con estados la verosimilitud marginal puede
depender de la semilla: conviene repetir la estimación antes de concluir (ejercicio 2).

¿Y el bloque? La misma comparación, con y sin la restricción sobre los términos de intercambio:

In [ ]:
bloque = {}
for spec in ["C-SV", "TVP-R1-SV"]:
    libre = estimate(spec, block=False)
    bloque[spec] = {"con bloque": models[spec].ml["logml"],
                    "sin bloque": libre.log_marginal_likelihood(n_draws=500, n_jobs=N_JOBS)["logml"]}
tabla = pd.DataFrame(bloque).T
tabla["log factor de Bayes del bloque"] = tabla["con bloque"] - tabla["sin bloque"]
tabla.round(1)

La exogeneidad de bloque gana unos nueve puntos de log verosimilitud marginal en los dos modelos
(8.8 y 9.7). Imponer lo que uno sabe, que el Perú no mueve los precios internacionales, quita
parámetros que los datos no necesitan.

Finalmente, la misma fecha y el mismo choque de una desviación estándar en los cuatro modelos.

In [ ]:
for m in models.values():
    if getattr(m, "irf", None) is None or "2024Q4" not in m.irf["draws"]:
        m.compute_irfs(dates=fechas, cred_interval=0.68, hor=21)

def irf_at(m, date, shock, response):
    t = [cd.quarter(d) for d in m.yy_dates].index(date)
    return m.irf["median"][t, :, names.index(response), names.index(shock)]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
for ax, (shock, response, title) in zip(axes, [("tot", "y", "PBI ante choque externo"),
                                               ("i", "pi", "Inflación ante choque de política")]):
    for spec, m in models.items():
        ax.plot(irf_at(m, "2024Q4", shock, response), lw=1.7, label=spec)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"{title}, 2024Q4")
    ax.set_xlabel("trimestres")
axes[0].legend()
plt.show()

Ante el choque externo, los tres modelos con volatilidad estocástica dan casi la misma respuesta
del PBI en 2024Q4; el modelo `C`, una mayor al principio. Ante el choque de política, la respuesta
de la inflación en `C` es entre siete y diez veces la de los demás: otra vez, el tamaño del choque de 1997-2002.
Lo que separa a los modelos es la varianza, no la dinámica.

## Ideas centrales

* **La volatilidad cambia, y mucho.** La desviación estándar del choque al PBI se multiplica por más
  de diez en 2020-21 y vuelve; la del choque de tasa cae desde fines de los noventa. La varianza
  constante es un promedio que no describe ningún período.
* **La volatilidad estocástica es lo que gana.** Frente al VAR de varianza constante, el `TVP-SV`
  mejora en más de 200 puntos de log verosimilitud marginal, pero el `C-SV` lo supera, con el
  `TVP-R1-SV` cerca: los coeficientes cambiantes no pagan su costo en estos datos.
* **El exterior es exógeno, y los datos lo confirman.** La exogeneidad de bloque de los términos de
  intercambio mejora la verosimilitud marginal en unos nueve puntos.
* **La transmisión también cambia, pero hay que normalizar.** Con un choque de una desviación
  estándar cambian a la vez el tamaño y la transmisión; con el mismo tamaño, la respuesta del PBI
  al choque externo de 2008 sigue siendo más del doble de la de 2024. Y en la R3, sin $A_t$ ni
  rezagos variables, las respuestas unitarias no cambian en el tiempo.
* **El peso del exterior depende de la especificación.** A 20 trimestres, el choque externo explica
  10 % de la varianza del PBI en el modelo `C` y entre 47 y 66 % en el `TVP-SV`.

## Ejercicios

1. **Pronósticos.** Reestime los cuatro modelos con datos hasta 2023Q4 y evalúe sus pronósticos de
   2024Q1 a 2026Q2 (`forecast`): error cuadrático medio de la mediana, ancho y cobertura de la banda
   al 68 %. ¿Qué modelo tiene la banda más honesta? ¿Coincide con el ganador de la sección 5?
2. **Semillas.** Repita la verosimilitud marginal de `C-SV`, `TVP-R1-SV` y `TVP-SV` con `seed=2`.
   ¿Se mantiene el orden?
3. **La R3 y la R2.** Estime `TVP-R3-SV` con `irf_1std=0` y compruebe que las respuestas son
   iguales en todas las fechas. Con `irf_1std=1`, ¿de dónde viene su variación? Añada `TVP-R2-SV`
   a la comparación.
4. **El bloque en el modelo completo.** Calcule el factor de Bayes de la exogeneidad de bloque en
   el `TVP-SV` (lea primero el aviso de tiempo).
5. **El orden.** Reordene el sistema con la tasa antes de la inflación y repita la superficie de la
   sección 3. ¿Cambia la fecha en que el choque externo pesó más?

## Referencias

- Chan, J.C.C. y Eisenstat, E. (2018). Bayesian model comparison for time-varying parameter VARs with stochastic volatility. *Journal of Applied Econometrics* 33(4), 509-532.
- Chan, J.C.C. y Jeliazkov, I. (2009). Efficient simulation and integrated likelihood estimation in state space models. *International Journal of Mathematical Modelling and Numerical Optimisation* 1(1-2), 101-120.
- Cushman, D.O. y Zha, T. (1997). Identifying monetary policy in a small open economy under flexible exchange rates. *Journal of Monetary Economics* 39(3), 433-448.
- Kim, S., Shephard, N. y Chib, S. (1998). Stochastic volatility: likelihood inference and comparison with ARCH models. *Review of Economic Studies* 65(3), 361-393.
- Primiceri, G.E. (2005). Time varying structural vector autoregressions and monetary policy. *Review of Economic Studies* 72(3), 821-852.
- Rodríguez, G., Vassallo, R. y Castillo B., P. (2023). Effects of external shocks on macroeconomic fluctuations in Pacific Alliance countries. *Economic Modelling* 124, 106302.
- Spiegelhalter, D.J., Best, N.G., Carlin, B.P. y van der Linde, A. (2002). Bayesian measures of model complexity and fit. *Journal of the Royal Statistical Society B* 64(4), 583-639.